In [1]:
```python
# ============================================================
# PROTEIN DATASET ANALYSIS USING NLP AND K-MEANS CLUSTERING
# Dataset: protein.csv
# Output: protein_clustered.csv
# ============================================================

# STEP 1: INSTALL LIBRARIES
%pip install pandas numpy matplotlib seaborn scikit-learn scipy

# STEP 2: IMPORT LIBRARIES
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_CLUSTERS = 3

# STEP 3: LOAD DATASET
df = pd.read_csv("protein.csv")

print("Original dataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())
display(df.head())

# STEP 4: CLEAN DATA
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")
df = df.drop_duplicates().reset_index(drop=True)

if df.empty:
    raise ValueError("The dataset is empty.")

print("\nDataset shape after cleaning:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

# STEP 5: IDENTIFY TEXT, SEQUENCE AND NUMERIC COLUMNS
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

text_cols = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

sequence_pattern = re.compile(
    r"(sequence|seq|amino_acid|aminoacid|protein_sequence|residues)",
    re.IGNORECASE
)

sequence_cols = [
    col for col in text_cols
    if sequence_pattern.search(col)
]

nlp_cols = [
    col for col in text_cols
    if col not in sequence_cols
]

print("\nNumeric columns:", numeric_cols)
print("Sequence columns:", sequence_cols)
print("Text columns:", nlp_cols)

# STEP 6: TEXT AND SEQUENCE PREPROCESSING
def clean_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def clean_sequence(value):
    if pd.isna(value):
        return ""

    value = str(value).upper()
    value = re.sub(r"\s+", "", value)
    value = re.sub(r"[^A-Z]", "", value)

    return value


if nlp_cols:
    df["nlp_text"] = (
        df[nlp_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .apply(clean_text)
    )
else:
    df["nlp_text"] = ""

if sequence_cols:
    df["protein_sequence"] = (
        df[sequence_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .apply(clean_sequence)
    )
else:
    df["protein_sequence"] = ""

print("\nProcessed text examples:")
display(df[["nlp_text", "protein_sequence"]].head())

# STEP 7: CREATE NLP FEATURES USING TF-IDF
feature_matrices = []
feature_names = []

nlp_vectorizer = None
sequence_vectorizer = None
nlp_features = None
sequence_features = None

# Natural-language text features
valid_text = df["nlp_text"].str.strip().ne("")

if valid_text.any():
    nlp_vectorizer = TfidfVectorizer(
        max_features=3000,
        stop_words="english",
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    nlp_features = nlp_vectorizer.fit_transform(df["nlp_text"])

    feature_matrices.append(nlp_features)
    feature_names.extend(
        ["nlp_" + t for t in nlp_vectorizer.get_feature_names_out()]
    )

    print("\nNLP feature matrix:", nlp_features.shape)

# Protein sequence features using character n-grams
valid_sequences = df["protein_sequence"].str.len().gt(0)

if valid_sequences.any():
    sequence_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 4),
        max_features=3000,
        sublinear_tf=True
    )

    sequence_features = sequence_vectorizer.fit_transform(
        df["protein_sequence"]
    )

    feature_matrices.append(sequence_features)
    feature_names.extend(
        ["seq_" + t
         for t in sequence_vectorizer.get_feature_names_out()]
    )

    print("Protein sequence feature matrix:", sequence_features.shape)

# STEP 8: PROCESS NUMERICAL FEATURES
numeric_feature_cols = [
    col for col in numeric_cols
    if not re.search(
        r"(^id$|_id$|^index$|^unnamed)",
        col,
        re.IGNORECASE
    )
]

if numeric_feature_cols:
    numeric_data = df[numeric_feature_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    # Remove columns containing only missing values
    numeric_feature_cols = [
        col for col in numeric_feature_cols
        if numeric_data[col].notna().any()
    ]

    if numeric_feature_cols:
        numeric_data = numeric_data[numeric_feature_cols]

        imputer = SimpleImputer(strategy="median")
        numeric_imputed = imputer.fit_transform(numeric_data)

        scaler = StandardScaler()
        numeric_scaled = scaler.fit_transform(numeric_imputed)

        feature_matrices.append(csr_matrix(numeric_scaled))
        feature_names.extend(
            ["num_" + col for col in numeric_feature_cols]
        )

        print("Numeric feature matrix:", numeric_scaled.shape)

# STEP 9: COMBINE ALL FEATURES
if not feature_matrices:
    raise ValueError(
        "No usable features found. Check whether the CSV contains "
        "text, protein sequences, or numerical columns."
    )

X = hstack(feature_matrices, format="csr", dtype=np.float64)

print("\nFinal feature matrix shape:", X.shape)

# STEP 10: REMOVE SAMPLES WITH NO USABLE FEATURES
# A row containing only missing/empty data cannot be clustered meaningfully.
valid_rows = np.asarray(X.getnnz(axis=1)).ravel() > 0

if not valid_rows.all():
    print("Rows with no usable features removed:", (~valid_rows).sum())
    df = df.loc[valid_rows].reset_index(drop=True)
    X = X[valid_rows]

if len(df) < 2:
    raise ValueError("At least two usable rows are required.")

# STEP 11: APPLY K-MEANS CLUSTERING
k = min(N_CLUSTERS, len(df))

kmeans = KMeans(
    n_clusters=k,
    random_state=RANDOM_STATE,
    n_init=10,
    max_iter=300
)

df["cluster"] = kmeans.fit_predict(X)

print("\nK-Means clustering completed.")
print("\nCluster counts:")
print(df["cluster"].value_counts().sort_index())

# STEP 12: EVALUATE CLUSTERING
labels = df["cluster"].to_numpy()
n_unique_clusters = len(np.unique(labels))

if 1 < n_unique_clusters < len(df):
    score = silhouette_score(
        X,
        labels,
        sample_size=min(2000, len(df)),
        random_state=RANDOM_STATE
    )

    print(f"\nSilhouette Score: {score:.4f}")
else:
    print("\nSilhouette score cannot be calculated.")

# STEP 13: VISUALIZE CLUSTERS IN TWO DIMENSIONS
if X.shape[0] >= 3 and X.shape[1] >= 2:
    svd = TruncatedSVD(
        n_components=2,
        random_state=RANDOM_STATE
    )

    X_2d = svd.fit_transform(X)

    plot_df = pd.DataFrame({
        "Component 1": X_2d[:, 0],
        "Component 2": X_2d[:, 1],
        "Cluster": labels.astype(str)
    })

    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        data=plot_df,
        x="Component 1",
        y="Component 2",
        hue="Cluster",
        palette="tab10",
        s=70
    )

    plt.title("Protein Dataset: NLP and K-Means Clustering")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(title="Cluster")
    plt.tight_layout()
    plt.show()

# STEP 14: ELBOW METHOD
max_k = min(10, len(df) - 1)

if max_k >= 2:
    k_values = range(1, max_k + 1)
    inertias = []

    for cluster_count in k_values:
        model = KMeans(
            n_clusters=cluster_count,
            random_state=RANDOM_STATE,
            n_init=10
        )

        model.fit(X)
        inertias.append(model.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(list(k_values), inertias, marker="o")
    plt.title("Elbow Method: Choosing the Number of Clusters")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Inertia")
    plt.xticks(list(k_values))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# STEP 15: DISPLAY CLUSTER SUMMARIES
print("\nCluster distribution:")
display(
    df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(name="Number of Records")
)

for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]

    print(f"\n{'=' * 50}")
    print(f"CLUSTER {cluster_id}")
    print("Number of records:", len(subset))

    text_examples = subset.loc[
        subset["nlp_text"].str.len() > 0, "nlp_text"
    ].head(3)

    if not text_examples.empty:
        print("\nText examples:")
        for example in text_examples:
            print("-", example[:200])

    sequence_examples = subset.loc[
        subset["protein_sequence"].str.len() > 0,
        "protein_sequence"
    ].head(3)

    if not sequence_examples.empty:
        print("\nSequence examples:")
        for example in sequence_examples:
            print("-", example[:100])

# STEP 16: DISPLAY TOP NLP TERMS PER CLUSTER
if nlp_vectorizer is not None and nlp_features is not None:
    terms = nlp_vectorizer.get_feature_names_out()

    for cluster_id in sorted(df["cluster"].unique()):
        mask = df["cluster"].to_numpy() == cluster_id

        mean_scores = np.asarray(
            nlp_features[mask].mean(axis=0)
        ).ravel()

        top_indices = mean_scores.argsort()[::-1][:10]

        print(f"\nCluster {cluster_id}: Top NLP terms")

        for idx in top_indices:
            print(f"{terms[idx]}: {mean_scores[idx]:.4f}")

# STEP 17: SAVE RESULTS
output_file = "protein_clustered.csv"

df.to_csv(output_file, index=False)

print("\nAnalysis completed successfully.")
print("Output saved as:", output_file)

display(df.head(10))
```


SyntaxError: invalid syntax (2814243658.py, line 1)